# 01 - Dataset Loading

**Project:** GroundedNutriRec  
**Student:** Student 1 - Data + Sequential Recommendation Lead  
**Scope:** Load and inspect the Food.com (RAW_recipes.csv, RAW_interactions.csv) and RecipeNLG datasets.  
**Goal:** Verify schemas, parse structured columns, and create 20K development samples.

## 1. Environment Setup

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path so that src modules are importable
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from SRC.data_preprocessing import (
    load_raw_recipes,
    load_raw_interactions,
    load_recipenlg,
    create_development_sample,
    SAMPLE_DATA_DIR,
    RAW_DATA_DIR,
    NUTRITION_COLUMN_NAMES,
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)
print(f'Project Root: {project_root}')
print(f'Raw Data Dir: {RAW_DATA_DIR}')
print(f'pandas {pd.__version__}, numpy {np.__version__}')

Project Root: C:\Users\Kush Shah\OneDrive\Desktop\Internship
Raw Data Dir: C:\Users\Kush Shah\OneDrive\Desktop\Internship\DATA\RAW
pandas 3.0.0, numpy 2.4.1


## 2. Load RAW_recipes.csv

Food.com dataset with 231,637 recipes. Columns include stringified Python lists  
for `tags`, `ingredients`, `steps`, and `nutrition` (a 7-element PDV vector).  
The `load_raw_recipes` function parses these lists and decomposes the nutrition vector.

In [2]:
# Load with full parsing: stringified lists -> Python lists, nutrition -> 7 columns
recipes_df = load_raw_recipes(parse_lists=True, parse_nutrition=True)
print(f'Shape: {recipes_df.shape}')
print(f'Columns: {list(recipes_df.columns)}')

Shape: (231637, 19)
Columns: ['name', 'id', 'minutes', 'contributor_id', 'submitted', 'tags', 'nutrition', 'n_steps', 'steps', 'description', 'ingredients', 'n_ingredients', 'calories', 'total_fat_pdv', 'sugar_pdv', 'sodium_pdv', 'protein_pdv', 'saturated_fat_pdv', 'carbohydrates_pdv']


In [3]:
recipes_df.head(3)

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients,calories,total_fat_pdv,sugar_pdv,sodium_pdv,protein_pdv,saturated_fat_pdv,carbohydrates_pdv
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"[60-minutes-or-less, time-to-make, course, main-ingredient, cuisine, prepara...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"[make a choice and proceed with recipe, depending on size of squash , cut in...",autumn is my favorite time of year to cook! this recipe \r\ncan be prepared ...,"[winter squash, mexican seasoning, mixed spice, honey, butter, olive oil, salt]",7,51.5,0.0,13.0,0.0,2.0,0.0,4.0
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"[30-minutes-or-less, time-to-make, course, main-ingredient, cuisine, prepara...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"[preheat oven to 425 degrees f, press dough into the bottom and sides of a 1...",this recipe calls for the crust to be prebaked a bit before adding ingredien...,"[prepared pizza crust, sausage patty, eggs, milk, salt and pepper, cheese]",6,173.4,18.0,0.0,17.0,22.0,35.0,1.0
2,all in the kitchen chili,112140,130,196586,2005-02-25,"[time-to-make, course, preparation, main-dish, chili, crock-pot-slow-cooker,...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"[brown ground beef in large pot, add chopped onions to ground beef when almo...",this modified version of 'mom's' chili was a hit at our 2004 christmas party...,"[ground beef, yellow onions, diced tomatoes, tomato paste, tomato soup, rote...",13,269.8,22.0,32.0,48.0,39.0,27.0,5.0


In [4]:
recipes_df.dtypes

name                            str
id                            int64
minutes                       int64
contributor_id                int64
submitted            datetime64[us]
tags                         object
nutrition                    object
n_steps                       int64
steps                        object
description                     str
ingredients                  object
n_ingredients                 int64
calories                    float64
total_fat_pdv               float64
sugar_pdv                   float64
sodium_pdv                  float64
protein_pdv                 float64
saturated_fat_pdv           float64
carbohydrates_pdv           float64
dtype: object

In [5]:
# Verify that nutrition was correctly decomposed into 7 separate float columns
for col in NUTRITION_COLUMN_NAMES:
    assert col in recipes_df.columns, f'Missing nutrition column: {col}'
    assert recipes_df[col].dtype in ['float64', 'float32'], f'{col} is not numeric'

print('Nutrition decomposition verified successfully.')
recipes_df[NUTRITION_COLUMN_NAMES].describe()

Nutrition decomposition verified successfully.


,calories,total_fat_pdv,sugar_pdv,sodium_pdv,protein_pdv,saturated_fat_pdv,carbohydrates_pdv
count,231637.000000,231637.00000,231637.000000,231637.000000,231637.00000,231637.000000,231637.000000
mean,473.942425,36.08070,84.296865,30.147485,34.68186,45.589150,15.560403
std,1189.711374,77.79884,800.080897,131.961589,58.47248,98.235758,81.824560
min,0.000000,0.00000,0.000000,0.000000,0.00000,0.000000,0.000000
25%,174.400000,8.00000,9.000000,5.000000,7.00000,7.000000,4.000000
50%,313.400000,20.00000,25.000000,14.000000,18.00000,23.000000,9.000000
75%,519.700000,41.00000,68.000000,33.000000,51.00000,52.000000,16.000000
max,434360.200000,17183.00000,362729.000000,29338.000000,6552.00000,10395.000000,36098.000000


In [6]:
# Check for null values across all columns
null_counts = recipes_df.isnull().sum()
print('Columns with null values:')
print(null_counts[null_counts > 0])

Columns with null values:
name              1
description    4979
dtype: int64


In [7]:
# Verify parsed list columns - each should contain Python lists, not strings
for col in ['tags', 'ingredients', 'steps']:
    sample_value = recipes_df[col].dropna().iloc[0]
    assert isinstance(sample_value, list), f'{col} should be a list, got {type(sample_value)}'
    print(f'{col}: type={type(sample_value).__name__}, sample_length={len(sample_value)}')

print('\nAll list columns parsed correctly.')

tags: type=list, sample_length=20
ingredients: type=list, sample_length=7
steps: type=list, sample_length=11

All list columns parsed correctly.


## 3. Load RAW_interactions.csv

User-recipe interaction data: `user_id`, `recipe_id`, `date`, `rating`, `review`.  
Ratings range from 0 to 5 where 0 means no explicit rating was given.

In [8]:
interactions_df = load_raw_interactions()
print(f'Shape: {interactions_df.shape}')
print(f'Columns: {list(interactions_df.columns)}')

Shape: (1132367, 5)
Columns: ['user_id', 'recipe_id', 'date', 'rating', 'review']


In [9]:
interactions_df.head(3)

,user_id,recipe_id,date,rating,review
0,38094,40893,2003-02-17,4,Great with a salad. Cooked on top of stove for 15 minutes.Added a shake of c...
1,1293707,40893,2011-12-21,5,"So simple, so delicious! Great for chilly fall evening. Should have doubled ..."
2,8937,44394,2002-12-01,4,This worked very well and is EASY. I used not quite a whole package (10oz) ...


In [10]:
interactions_df.dtypes

user_id               int64
recipe_id             int64
date         datetime64[us]
rating                int64
review                  str
dtype: object

In [11]:
# Rating distribution - critical for implicit feedback derivation
print('Rating value counts:')
print(interactions_df['rating'].value_counts().sort_index())
print(f'\nRating=0 (no explicit feedback): {(interactions_df["rating"] == 0).sum():,}')

Rating value counts:
rating
0     60847
1     12818
2     14123
3     40855
4    187360
5    816364
Name: count, dtype: int64

Rating=0 (no explicit feedback): 60,847


In [12]:
# Unique user and recipe counts
unique_users = interactions_df['user_id'].nunique()
unique_recipes_interacted = interactions_df['recipe_id'].nunique()
total_interactions = len(interactions_df)
sparsity = 1 - (total_interactions / (unique_users * unique_recipes_interacted))

print(f'Unique users:    {unique_users:,}')
print(f'Unique recipes:  {unique_recipes_interacted:,}')
print(f'Total interactions: {total_interactions:,}')
print(f'Interaction matrix sparsity: {sparsity:.6f} ({sparsity*100:.4f}%)')

Unique users:    226,570
Unique recipes:  231,637
Total interactions: 1,132,367
Interaction matrix sparsity: 0.999978 (99.9978%)


## 4. Load RecipeNLG (Parquet)

RecipeNLG: 2.2M recipes scraped from recipe websites.  
Used as a supplementary corpus for ingredient vocabulary and NER training.  
Loading first rows only to inspect schema without loading 1 GB into memory.

In [13]:
# Load first 5 rows to inspect schema without loading 1 GB into memory
recipenlg_sample = load_recipenlg(nrows=5)
print(f'Columns: {list(recipenlg_sample.columns)}')
recipenlg_sample.head()

Columns: ['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source', 'NER']


,Unnamed: 0,title,ingredients,directions,link,source,NER
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. evaporated milk"", ""1/2 tsp. vanil...","[""In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and bu...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""butter"", ""bite size shredded ric..."
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned chicken breasts"", ""1 can cream...","[""Place chipped beef on bottom of baking dish."", ""Place chicken on top of be...",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom soup"", ""sour cream""]"
2,2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg. cream cheese, cubed"", ""1/3 c...","[""In a slow cooker, combine all ingredients. Cover and cook on low for 4 hou...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""garlic powder"", ""salt"", ""pepper""]"
3,3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans chicken gravy"", ""1 (10 1/2 oz...","[""Boil and debone chicken."", ""Put bite size pieces in average size square ca...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroom soup"", ""shredded cheese""]"
4,4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker crumbs"", ""1 c. melted butter"",...","[""Combine first four ingredients and press in 13 x 9-inch ungreased pan."", ""...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""butter"", ""powdered sugar"", ""choc..."


## 5. Create Development Samples

20,000-row random samples saved to `data/sample/` for rapid  
iteration during EDA and model prototyping.

In [14]:
SAMPLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

recipes_sample = create_development_sample(
    recipes_df,
    sample_size=20000,
    random_seed=42,
    output_path=SAMPLE_DATA_DIR / 'recipes_sample.parquet',
)
print(f'Recipes sample shape: {recipes_sample.shape}')
print(f'Saved to: {SAMPLE_DATA_DIR / "recipes_sample.parquet"}')

Recipes sample shape: (20000, 19)
Saved to: C:\Users\Kush Shah\OneDrive\Desktop\Internship\DATA\SAMPLE\recipes_sample.parquet


In [15]:
interactions_sample = create_development_sample(
    interactions_df,
    sample_size=20000,
    random_seed=42,
    output_path=SAMPLE_DATA_DIR / 'interactions_sample.parquet',
)
print(f'Interactions sample shape: {interactions_sample.shape}')
print(f'Saved to: {SAMPLE_DATA_DIR / "interactions_sample.parquet"}')

Interactions sample shape: (20000, 5)
Saved to: C:\Users\Kush Shah\OneDrive\Desktop\Internship\DATA\SAMPLE\interactions_sample.parquet


## 6. Summary

| Dataset | File | Rows | Columns | Key Features |
| --- | --- | --- | --- | --- |
| Food.com Recipes | RAW_recipes.csv | ~231K | 12 + 7 nutrition | Parsed lists, nutrition decomposed |
| Food.com Interactions | RAW_interactions.csv | ~1.1M | 5 | date parsed, rating distribution inspected |
| RecipeNLG | recipenlg.parquet | ~2.2M | varies | Supplementary corpus for NER |

**Next:** `02_EDA_FOOD_DATASET.ipynb` for detailed exploratory analysis.